In [1]:
import concurrent.futures
import os
import pandas as pd
from tqdm import tqdm
import scanpy as sc
if not os.path.exists('../../../data/rna/pseudobulk/outputs'):
    os.makedirs('../../../data/rna/pseudobulk/outputs')

## PBMC

### 1. Create psuedo counts

In [2]:
%run ../../00-utilities/functions/python/aggregate_counts.py

In [3]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-pbmc-raw.h5ad')

In [4]:
groups = adata.obs["sample.sampleKitGuid"].unique().tolist()
adata_dict = {group: adata[adata.obs["sample.sampleKitGuid"] == group].copy() for group in groups}

In [5]:
results_dir  = '../../../data/rna/pseudobulk/outputs'
with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
    tasks = [(sample_id, adata, 'aifi_plot_l3', results_dir, 'pbmc') for sample_id, adata in adata_dict.items()]
    list(tqdm(executor.map(process_sample_wrapper, tasks), total=len(tasks)))

100%|██████████| 353/353 [07:10<00:00,  1.22s/it]


### 2. Create a metadata file for DEseq2

In [6]:
# Compute per-sample × celltype cell counts, plus total cells per sample
cell_counts = (
    # Count cells per (sample, celltype)
    adata.obs.groupby(
        ["sample.sampleKitGuid", "aifi_plot_l3"], observed=True)
    .size()
    .reset_index(name="n_cells")
    # Merge with total cells per sample
    .merge(
        adata.obs.groupby("sample.sampleKitGuid", observed=True)
        .size()
        .reset_index(name="total_cells"),
        on="sample.sampleKitGuid",
        how="left"
    )
)

# Add a flag column indicating whether a sample–celltype passes thresholds:
#   - at least 10 cells in that sample × celltype
#   - at least 1000 total cells in the sample overall
cell_counts["keep"] = (
    (cell_counts["n_cells"] >= 10) & (cell_counts["total_cells"] >= 1000)
)

In [7]:
# Sample-level metadata
meta = adata.obs[
    [
        "sample.sampleKitGuid",
        "sample.visitDetails",
        "sample.visitName",
        "label.visitDetails",
        "label.visitName",
        "sample.diseaseStatesRecordedAtVisit",
        "subject.biologicalSex",
        "sample.drawDate",
        "subject.birthYear",
        "subject.age",
        "subject.ethnicity",
        "subject.race",
        "subject.subjectGuid",
        "subject.cmv",
        "cohort.cohortGuid",
        "manual.time_stamp",
        "manual.category",
        "manual.flu_response"
    ]
].drop_duplicates()

# Merge counts into metadata
result = meta.merge(cell_counts, on="sample.sampleKitGuid", how="left")
result.to_csv("../../../data/rna/pseudobulk/outputs/pbmc_sample_kit_metadata.csv")

## 3. Filter gene set creation

In [9]:
%run ../../00-utilities/functions/python/filter_genes.py

In [10]:
adata.obs['label.visitDetails'].value_counts()

Healthy    529853
PreTx      257359
PI2C       231465
EI         223629
ASCT60d    194404
ASCT1y     171476
ASCT2y      93044
Name: label.visitDetails, dtype: int64

#### Treatment timepoints

In [11]:
visit_pairs = [
    ("PreTx", "PI2C"),
    ("PreTx", "EI"),
    ("PI2C", "EI"),
    ("EI", "ASCT60d"),
    ("EI", "ASCT1y"),
    ("EI", "ASCT2y"),
    ("ASCT60d", "ASCT1y"),
    ("ASCT1y", "ASCT2y"),
]

for v1, v2 in visit_pairs:
    # keep only the two visits
    flt = adata.obs["label.visitDetails"].isin([v1, v2])
    adata_filtered = adata[flt].copy()

    out = f"../../../data/rna/pseudobulk/outputs/pbmc_l3_{v1}-vs-{v2}_filtered_gene_list.csv"

    # run filtering
    filtered_gene_df = get_filtered_genes_by_label(
        adata_filtered,
        celltype_col="aifi_plot_l3",
        min_frac=0.1,
        output_csv=out,
    )

    print(f"Saved {v1} vs {v2} at {out}")


Saved PreTx vs PI2C at outputs/pbmc_l3_PreTx-vs-PI2C_filtered_gene_list.csv
Saved PreTx vs EI at outputs/pbmc_l3_PreTx-vs-EI_filtered_gene_list.csv
Saved PI2C vs EI at outputs/pbmc_l3_PI2C-vs-EI_filtered_gene_list.csv
Saved EI vs ASCT60d at outputs/pbmc_l3_EI-vs-ASCT60d_filtered_gene_list.csv
Saved EI vs ASCT1y at outputs/pbmc_l3_EI-vs-ASCT1y_filtered_gene_list.csv
Saved EI vs ASCT2y at outputs/pbmc_l3_EI-vs-ASCT2y_filtered_gene_list.csv
Saved ASCT60d vs ASCT1y at outputs/pbmc_l3_ASCT60d-vs-ASCT1y_filtered_gene_list.csv
Saved ASCT1y vs ASCT2y at outputs/pbmc_l3_ASCT1y-vs-ASCT2y_filtered_gene_list.csv


#### PBMC Healthy Comparisons

In [12]:
visits = [
    "PreTx",
    "PI2C",
    "EI",
    "ASCT60d",
    "ASCT1y",
    "ASCT2y"
]

for visit in visits:
    # keep healthy + current visit
    flt = adata.obs["label.visitDetails"].isin(["Healthy", visit])
    adata_filtered = adata[flt].copy()

    out = f"../../../data/rna/pseudobulk/outputs/pbmc_l3_healthy-vs-{visit}_filtered_gene_list.csv"

    # run filtering
    filtered_gene_df = get_filtered_genes_by_label(
        adata_filtered,
        celltype_col="aifi_plot_l3",
        min_frac=0.1,
        output_csv=out,
    )

    print(f"Saved Healthy vs {visit} at {out}")

Saved Healthy vs PreTx at outputs/pbmc_l3_healthy-vs-PreTx_filtered_gene_list.csv
Saved Healthy vs PI2C at outputs/pbmc_l3_healthy-vs-PI2C_filtered_gene_list.csv
Saved Healthy vs EI at outputs/pbmc_l3_healthy-vs-EI_filtered_gene_list.csv
Saved Healthy vs ASCT60d at outputs/pbmc_l3_healthy-vs-ASCT60d_filtered_gene_list.csv
Saved Healthy vs ASCT1y at outputs/pbmc_l3_healthy-vs-ASCT1y_filtered_gene_list.csv
Saved Healthy vs ASCT2y at outputs/pbmc_l3_healthy-vs-ASCT2y_filtered_gene_list.csv
